# ID Forgery Detection — Qwen2-VL-2B + LoRA

This is the corrected one-file Google Colab notebook.

**Important fix:** Qwen2-VL was failing with:

`Mismatch in image token count ... Likely due to truncation='max_length'`

The training pipeline below **never uses `truncation=True` or `max_length` in the multimodal processor call**. Padding is handled safely by a custom collator.

Run cells from top to bottom on a **T4 GPU**.


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not detected. Go to Runtime → Change runtime type → select T4 GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)

PyTorch: 2.9.0+cu128
CUDA available: True
GPU: Tesla T4
CUDA: 12.8


In [ ]:
# ============================================================
# CELL 2 — COMPATIBLE ENVIRONMENT
# ============================================================

import subprocess
import sys

packages = [
    # PyTorch pair
    "torch==2.9.0",
    "torchvision==0.24.0",
    "torchaudio==2.9.0",

    # Hugging Face
    "transformers==4.56.2",
    "peft==0.20.0",
    "accelerate==1.10.1",
    "bitsandbytes==0.46.1",

    # Dataset / utilities
    "datasets==4.0.0",
    "huggingface_hub>=0.34.0",
    "qwen-vl-utils",
    "pillow==11.3.0",
    "opencv-python-headless",
    "tqdm",
    "scikit-learn",
    "pyarrow",
]

cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "--no-cache-dir",
]

# Install PyTorch from the official CUDA 12.6 index.
subprocess.check_call(
    cmd
    + [
        "torch==2.9.0",
        "torchvision==0.24.0",
        "torchaudio==2.9.0",
        "--index-url",
        "https://download.pytorch.org/whl/cu126",
    ]
)

# Install the rest from PyPI.
subprocess.check_call(
    cmd + packages[3:]
)

print()
print("=" * 60)
print("INSTALLATION COMPLETE")
print("=" * 60)
print()
print("IMPORTANT:")
print("Runtime → Restart session")
print()
print("After restarting, continue from CELL 3.")


INSTALLATION COMPLETE

IMPORTANT:
Runtime → Restart session

After restarting, continue from CELL 3.


In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

import torchvision

print("Torchvision:", torchvision.__version__)

from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model

print("================================")
print("ALL IMPORTS WORKING ✅")
print("================================")

Torch: 2.9.0+cu128
CUDA: 12.8
GPU: Tesla T4
Torchvision: 0.24.0+cu128
ALL IMPORTS WORKING ✅


In [ ]:
# ============================================================
# CELL 3 — VERIFY ENVIRONMENT
# ============================================================

import torch
import torchvision
import transformers
import peft
import accelerate
import bitsandbytes

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)

print("Python:", __import__("sys").version)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("BitsAndBytes:", bitsandbytes.__version__)

print()
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print()
print("=" * 60)
print("TESTING QWEN + PEFT IMPORTS")
print("=" * 60)

from transformers import (
    AutoProcessor,
    Qwen2VLForConditionalGeneration,
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

print()
print("ALL IMPORTS WORKING ✅")

ENVIRONMENT
Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch: 2.9.0+cu128
Torchvision: 0.24.0+cu128
Transformers: 4.56.2
PEFT: 0.20.0
Accelerate: 1.10.1
BitsAndBytes: 0.46.1

CUDA available: True
CUDA version: 12.8
GPU: Tesla T4

TESTING QWEN + PEFT IMPORTS

ALL IMPORTS WORKING ✅


In [ ]:
# ============================================================
# CELL 4 — PROJECT CONFIGURATION
# ============================================================

import io
import json
import random
import re
import shutil
from pathlib import Path

import cv2
import numpy as np
import torch

from PIL import Image, ImageDraw, ImageFont
from tqdm.auto import tqdm


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

PROJECT = Path("/content/id-forgery-detector")

RAW_DIR = PROJECT / "data" / "raw"
FORGED_DIR = PROJECT / "data" / "forged"
DATASET_DIR = PROJECT / "data" / "dataset"
OUTPUT_DIR = PROJECT / "outputs"
LORA_DIR = OUTPUT_DIR / "lora_weights"

for directory in [
    RAW_DIR,
    FORGED_DIR,
    DATASET_DIR,
    OUTPUT_DIR,
    LORA_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

TARGET_IMAGE_COUNT = 100
TRAIN_SPLIT = 0.85


# ------------------------------------------------------------
# LoRA
# ------------------------------------------------------------

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

NUM_EPOCHS = 3
BATCH_SIZE = 1
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4


# ------------------------------------------------------------
# Question
# ------------------------------------------------------------

QUESTION = (
    "Is this identity document authentic or forged? "
    "Explain your reasoning briefly."
)


DOC_TYPES = [
    "alb_id",
    "aze_passport",
    "esp_id",
    "est_id",
    "fin_id",
    "grc_passport",
    "lva_passport",
    "rus_internalpassport",
    "svk_id",
    "srb_passport",
]

FORGERY_METHODS = [
    "text_replace",
    "region_copy",
    "blur_recompress",
]


print("Project:", PROJECT)
print("Model:", MODEL_ID)

Project: /content/id-forgery-detector
Model: Qwen/Qwen2-VL-2B-Instruct


In [ ]:
# ============================================================
# CELL 5 — DOWNLOAD AUTHENTIC IMAGES
# ============================================================

from datasets import load_dataset


HF_DATASETS = [
    "ruturajnawale/midv500",
    "Noaman/midv500",
]


def to_pil(img):

    if img is None:
        return None

    if isinstance(img, Image.Image):
        return img.convert("RGB")

    if isinstance(img, dict):

        if img.get("bytes") is not None:
            return Image.open(
                io.BytesIO(img["bytes"])
            ).convert("RGB")

        if img.get("path"):
            return Image.open(
                img["path"]
            ).convert("RGB")

    if isinstance(img, bytes):
        return Image.open(
            io.BytesIO(img)
        ).convert("RGB")

    return None


def find_image(row):

    # Most common names
    for key in [
        "image",
        "img",
        "pixel_values",
    ]:

        if key in row:

            img = to_pil(row[key])

            if img is not None:
                return img

    # Search all columns
    for key, value in row.items():

        if (
            "image" in key.lower()
            or "pixel" in key.lower()
        ):

            img = to_pil(value)

            if img is not None:
                return img

    return None


def download_from_hf(repo, count):

    print()
    print("Loading:", repo)

    ds = load_dataset(
        repo,
        split="train",
    )

    print("Dataset size:", len(ds))

    rows = list(ds)

    random.shuffle(rows)

    rows = rows[:count]

    manifest = []

    for i, row in enumerate(
        tqdm(
            rows,
            desc=f"Downloading {repo}",
        )
    ):

        img = find_image(row)

        if img is None:
            continue

        doc_type = row.get(
            "doc_type",
            DOC_TYPES[i % len(DOC_TYPES)],
        )

        path = RAW_DIR / f"{doc_type}_{i:04d}.jpg"

        img.save(
            path,
            quality=95,
        )

        manifest.append(
            {
                "id": f"real_{i:04d}",
                "image": str(
                    path.relative_to(PROJECT)
                ),
                "label": "authentic",
                "doc_type": doc_type,
                "source": repo,
            }
        )

    return manifest


# Clean old images
for p in RAW_DIR.glob("*"):

    if p.is_file():
        p.unlink()


manifest = []


for repo in HF_DATASETS:

    if len(manifest) >= TARGET_IMAGE_COUNT:
        break

    try:

        remaining = (
            TARGET_IMAGE_COUNT
            - len(manifest)
        )

        downloaded = download_from_hf(
            repo,
            remaining,
        )

        manifest.extend(downloaded)

        print(
            repo,
            "→",
            len(downloaded),
            "images",
        )

    except Exception as e:

        print(
            "FAILED:",
            repo,
        )

        print(
            type(e).__name__,
            str(e),
        )


if len(manifest) == 0:

    raise RuntimeError(
        "No authentic images were downloaded."
    )


manifest_path = (
    RAW_DIR.parent
    / "raw_manifest.json"
)

manifest_path.write_text(
    json.dumps(
        manifest,
        indent=2,
    )
)

print()
print(
    "Authentic images:",
    len(manifest),
)


Loading: ruturajnawale/midv500


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset size: 237


ruturajnawale/midv500 → 100 images

Authentic images: 100


In [ ]:
# ============================================================
# CELL 6 — GENERATE FORGED IMAGES
# ============================================================

FAKE_NAMES = [
    "JOHN DOE",
    "JANE SMITH",
    "ALEX RIVERA",
    "MARIA GARCIA",
    "LI WEI",
]

FAKE_DATES = [
    "01/01/1990",
    "15/06/1985",
    "22/03/1978",
    "09/11/1995",
    "30/12/2000",
]

FAKE_IDS = [
    "X1234567",
    "AB987654",
    "ID000042",
    "ZZ445566",
    "DOC778899",
]


FIELD_LABELS = {
    "name": FAKE_NAMES,
    "date": FAKE_DATES,
    "id_number": FAKE_IDS,
}


EXPLANATIONS = {

    "text_replace":
        "Forged — the {field} field shows inconsistent "
        "text rendering, font, and alignment.",

    "region_copy":
        "Forged — the {field} region shows a "
        "copy-paste and lighting inconsistency.",

    "blur_recompress":
        "Forged — the {field} region shows blur and "
        "JPEG compression artifacts consistent with editing.",
}


def random_field():

    return random.choice(
        list(FIELD_LABELS.keys())
    )


def field_bbox(
    width,
    height,
    field,
):

    regions = {

        "name": (
            int(width * 0.25),
            int(height * 0.55),
            int(width * 0.75),
            int(height * 0.62),
        ),

        "date": (
            int(width * 0.25),
            int(height * 0.65),
            int(width * 0.55),
            int(height * 0.72),
        ),

        "id_number": (
            int(width * 0.25),
            int(height * 0.75),
            int(width * 0.70),
            int(height * 0.82),
        ),
    }

    return regions[field]


# ------------------------------------------------------------
# Method 1 — Text replacement
# ------------------------------------------------------------

def forge_text_replace(
    img,
    field,
):

    height, width = img.shape[:2]

    x1, y1, x2, y2 = field_bbox(
        width,
        height,
        field,
    )

    pil = Image.fromarray(
        cv2.cvtColor(
            img,
            cv2.COLOR_BGR2RGB,
        )
    )

    draw = ImageDraw.Draw(pil)

    patch = img[
        y1:y2,
        x1:x2,
    ]

    bg_color = tuple(
        int(c)
        for c in np.median(
            patch.reshape(-1, 3),
            axis=0,
        )
    )

    draw.rectangle(
        [
            x1,
            y1,
            x2,
            y2,
        ],
        fill=bg_color,
    )

    text = random.choice(
        FIELD_LABELS[field]
    )

    font = ImageFont.load_default()

    font_paths = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf",
    ]

    font_size = max(
        12,
        (y2 - y1) - 4,
    )

    for font_path in font_paths:

        try:

            font = ImageFont.truetype(
                font_path,
                font_size,
            )

            break

        except OSError:
            pass

    draw.text(
        (
            x1 + 2,
            y1 + 1,
        ),
        text,
        fill=(
            20,
            20,
            20,
        ),
        font=font,
    )

    return cv2.cvtColor(
        np.array(pil),
        cv2.COLOR_RGB2BGR,
    )


# ------------------------------------------------------------
# Method 2 — Region copy
# ------------------------------------------------------------

def forge_region_copy(
    img,
    field,
):

    height, width = img.shape[:2]

    x1, y1, x2, y2 = field_bbox(
        width,
        height,
        field,
    )

    patch_width = x2 - x1
    patch_height = y2 - y1

    max_x = max(
        0,
        width - patch_width - 1,
    )

    max_y = max(
        0,
        height - patch_height - 1,
    )

    source_x = random.randint(
        0,
        max_x,
    )

    source_y = random.randint(
        0,
        max_y,
    )

    patch = img[
        source_y:
        source_y + patch_height,

        source_x:
        source_x + patch_width,
    ].copy()

    output = img.copy()

    output[
        y1:y2,
        x1:x2,
    ] = patch

    return output


# ------------------------------------------------------------
# Method 3 — Blur + recompression
# ------------------------------------------------------------

def forge_blur_recompress(
    img,
    field,
):

    height, width = img.shape[:2]

    x1, y1, x2, y2 = field_bbox(
        width,
        height,
        field,
    )

    output = img.copy()

    region = output[
        y1:y2,
        x1:x2,
    ]

    blurred = cv2.GaussianBlur(
        region,
        (9, 9),
        0,
    )

    success, buffer = cv2.imencode(
        ".jpg",
        blurred,
        [
            cv2.IMWRITE_JPEG_QUALITY,
            15,
        ],
    )

    if not success:
        return output

    recompressed = cv2.imdecode(
        buffer,
        cv2.IMREAD_COLOR,
    )

    output[
        y1:y2,
        x1:x2,
    ] = recompressed

    return output


FORGE_FUNCTIONS = {

    "text_replace":
        forge_text_replace,

    "region_copy":
        forge_region_copy,

    "blur_recompress":
        forge_blur_recompress,
}


# Clean previous forged images

for p in FORGED_DIR.glob("*"):

    if p.is_file():
        p.unlink()


forged_manifest = []

images = sorted(
    RAW_DIR.glob("*.jpg")
)


for i, source in enumerate(
    tqdm(
        images,
        desc="Generating forgeries",
    )
):

    img = cv2.imread(
        str(source)
    )

    if img is None:
        continue

    method = FORGERY_METHODS[
        i % len(FORGERY_METHODS)
    ]

    field = random_field()

    forged = FORGE_FUNCTIONS[
        method
    ](
        img,
        field,
    )

    output_path = (
        FORGED_DIR
        / f"forged_{source.stem}_{method}.jpg"
    )

    cv2.imwrite(
        str(output_path),
        forged,
        [
            cv2.IMWRITE_JPEG_QUALITY,
            90,
        ],
    )

    forged_manifest.append(
        {
            "id": f"forged_{i:04d}",

            "image": str(
                output_path.relative_to(PROJECT)
            ),

            "label": "forged",

            "method": method,

            "field": field,

            "source_image": source.name,

            "answer": EXPLANATIONS[
                method
            ].format(
                field=field.replace(
                    "_",
                    " ",
                )
            ),
        }
    )


forged_manifest_path = (
    RAW_DIR.parent
    / "forged_manifest.json"
)

forged_manifest_path.write_text(
    json.dumps(
        forged_manifest,
        indent=2,
    )
)

print()
print(
    "Forged images:",
    len(forged_manifest),
)

Generating forgeries:   0%|          | 0/100 [00:00<?, ?it/s]


Forged images: 100


In [ ]:
# FIX: Remove broken torchvision and install a compatible version

import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "uninstall",
    "-y", "torchvision"
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-cache-dir",
    "--force-reinstall",
    "torchvision==0.24.0"
])

print("torchvision installed.")
print("RESTART THE RUNTIME NOW.")

torchvision installed.
RESTART THE RUNTIME NOW.


In [ ]:
# ============================================================
# CELL 7 — CREATE TRAIN / TEST JSONL
# ============================================================

real = json.loads(
    (
        RAW_DIR.parent
        / "raw_manifest.json"
    ).read_text()
)

forged = json.loads(
    (
        RAW_DIR.parent
        / "forged_manifest.json"
    ).read_text()
)


samples = []


# Authentic examples

for item in real:

    samples.append(
        {
            "image": item["image"],
            "question": QUESTION,
            "answer": "Authentic",
            "label": "authentic",
        }
    )


# Forged examples

for item in forged:

    samples.append(
        {
            "image": item["image"],
            "question": QUESTION,
            "answer": item["answer"],
            "label": "forged",
        }
    )


random.seed(
    RANDOM_SEED
)

random.shuffle(samples)


split_index = int(
    len(samples) * TRAIN_SPLIT
)

train_samples = samples[
    :split_index
]

test_samples = samples[
    split_index:
]


train_path = (
    DATASET_DIR
    / "train.jsonl"
)

test_path = (
    DATASET_DIR
    / "test.jsonl"
)


with open(
    train_path,
    "w",
) as f:

    for row in train_samples:

        f.write(
            json.dumps(row)
            + "\n"
        )


with open(
    test_path,
    "w",
) as f:

    for row in test_samples:

        f.write(
            json.dumps(row)
            + "\n"
        )


stats = {

    "total":
        len(samples),

    "train":
        len(train_samples),

    "test":
        len(test_samples),

    "authentic":
        sum(
            s["label"] == "authentic"
            for s in samples
        ),

    "forged":
        sum(
            s["label"] == "forged"
            for s in samples
        ),
}


(
    DATASET_DIR
    / "stats.json"
).write_text(
    json.dumps(
        stats,
        indent=2,
    )
)


print(
    json.dumps(
        stats,
        indent=2,
    )
)

{
  "total": 200,
  "train": 170,
  "test": 30,
  "authentic": 100,
  "forged": 100
}


In [ ]:
# ============================================================
# CELL 8 — LOAD QWEN2-VL + 4-BIT QUANTIZATION
# ============================================================

from transformers import (
    AutoProcessor,
    BitsAndBytesConfig,
    Qwen2VLForConditionalGeneration,
)


# ------------------------------------------------------------
# Select dtype
# ------------------------------------------------------------

if torch.cuda.is_bf16_supported():

    COMPUTE_DTYPE = torch.bfloat16

else:

    COMPUTE_DTYPE = torch.float16


print(
    "Compute dtype:",
    COMPUTE_DTYPE,
)


# ------------------------------------------------------------
# 4-bit configuration
# ------------------------------------------------------------

bnb_config = BitsAndBytesConfig(

    load_in_4bit=True,

    bnb_4bit_quant_type="nf4",

    bnb_4bit_compute_dtype=COMPUTE_DTYPE,

    bnb_4bit_use_double_quant=True,
)


print()
print("Loading:", MODEL_ID)


# ------------------------------------------------------------
# Model
# ------------------------------------------------------------

model = Qwen2VLForConditionalGeneration.from_pretrained(

    MODEL_ID,

    quantization_config=bnb_config,

    device_map="auto",

    torch_dtype=COMPUTE_DTYPE,
)


# ------------------------------------------------------------
# Processor
# ------------------------------------------------------------

processor = AutoProcessor.from_pretrained(

    MODEL_ID,

    use_fast=False,
)


# ------------------------------------------------------------
# Training settings
# ------------------------------------------------------------

model.config.use_cache = False


print()
print("MODEL LOADED ✅")

`torch_dtype` is deprecated! Use `dtype` instead!


Compute dtype: torch.bfloat16

Loading: Qwen/Qwen2-VL-2B-Instruct


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]


MODEL LOADED ✅


In [ ]:
# ============================================================
# CELL 9 — DATASET + QWEN2-VL COLLATOR
# ============================================================

from torch.utils.data import Dataset


class ForgeryDataset(Dataset):

    def __init__(
        self,
        jsonl_path,
    ):

        self.samples = []

        with open(
            jsonl_path
        ) as f:

            for line in f:

                self.samples.append(
                    json.loads(line)
                )


    def __len__(self):

        return len(self.samples)


    def __getitem__(
        self,
        index,
    ):

        sample = self.samples[index]

        image_path = (
            PROJECT
            / sample["image"]
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        return {
            "image": image,
            "question": sample["question"],
            "answer": sample["answer"],
        }


class QwenVLCollator:

    def __init__(
        self,
        processor,
    ):

        self.processor = processor


    def make_messages(
        self,
        example,
        include_answer=True,
    ):

        user_message = {

            "role": "user",

            "content": [

                {
                    "type": "image",
                    "image": example["image"],
                },

                {
                    "type": "text",
                    "text": example["question"],
                },

            ],
        }


        if not include_answer:

            return [
                user_message
            ]


        assistant_message = {

            "role": "assistant",

            "content": [

                {
                    "type": "text",
                    "text": example["answer"],
                }

            ],
        }


        return [
            user_message,
            assistant_message,
        ]


    def __call__(
        self,
        examples,
    ):

        full_texts = []
        prompt_texts = []
        images = []


        for example in examples:

            full_messages = self.make_messages(
                example,
                include_answer=True,
            )

            prompt_messages = self.make_messages(
                example,
                include_answer=False,
            )


            full_text = (
                self.processor.apply_chat_template(
                    full_messages,
                    tokenize=False,
                    add_generation_prompt=False,
                )
            )


            prompt_text = (
                self.processor.apply_chat_template(
                    prompt_messages,
                    tokenize=False,
                    add_generation_prompt=True,
                )
            )


            full_texts.append(
                full_text
            )

            prompt_texts.append(
                prompt_text
            )

            images.append(
                example["image"]
            )


        # ----------------------------------------------------
        # CRITICAL:
        #
        # NO truncation=True
        # NO max_length
        #
        # This keeps image placeholder tokens aligned
        # with image tensors.
        # ----------------------------------------------------

        batch = self.processor(

            text=full_texts,

            images=images,

            return_tensors="pt",

            padding=True,
        )


        # ----------------------------------------------------
        # Build labels
        # ----------------------------------------------------

        labels = (
            batch["input_ids"]
            .clone()
        )


        pad_token_id = (
            self.processor
            .tokenizer
            .pad_token_id
        )


        if pad_token_id is not None:

            labels[
                labels == pad_token_id
            ] = -100


        # ----------------------------------------------------
        # Mask the user prompt.
        #
        # We only want the model to learn the assistant
        # response, not reproduce the question.
        # ----------------------------------------------------

        for i, prompt_text in enumerate(
            prompt_texts
        ):

            prompt_tokens = self.processor(

                text=[
                    prompt_text
                ],

                images=[
                    images[i]
                ],

                return_tensors="pt",

                padding=False,
            )

            prompt_length = (
                prompt_tokens[
                    "input_ids"
                ].shape[1]
            )


            labels[
                i,
                :prompt_length
            ] = -100


        batch["labels"] = labels

        return batch


train_dataset = ForgeryDataset(
    DATASET_DIR / "train.jsonl"
)


collator = QwenVLCollator(
    processor
)


print(
    "Training examples:",
    len(train_dataset),
)

print(
    "Collator ready ✅"
)

Training examples: 170
Collator ready ✅


In [ ]:
# ============================================================
# CELL 10 — PREPARE QLORA
# ============================================================

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)


# ------------------------------------------------------------
# Prepare quantized model
# ------------------------------------------------------------

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)


model.gradient_checkpointing_enable()


# ------------------------------------------------------------
# LoRA configuration
# ------------------------------------------------------------

lora_config = LoraConfig(

    r=LORA_R,

    lora_alpha=LORA_ALPHA,

    lora_dropout=LORA_DROPOUT,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    task_type="CAUSAL_LM",
)


# ------------------------------------------------------------
# Attach LoRA
# ------------------------------------------------------------

model = get_peft_model(
    model,
    lora_config,
)


print()
print("=" * 60)
print("TRAINABLE PARAMETERS")
print("=" * 60)

model.print_trainable_parameters()


TRAINABLE PARAMETERS
trainable params: 18,464,768 || all params: 2,227,450,368 || trainable%: 0.8290


In [ ]:
# ============================================================
# CELL 11 — TRAIN
# ============================================================

from transformers import (
    Trainer,
    TrainingArguments,
)


training_args = TrainingArguments(

    output_dir=str(
        LORA_DIR
    ),

    num_train_epochs=NUM_EPOCHS,

    per_device_train_batch_size=BATCH_SIZE,

    gradient_accumulation_steps=GRAD_ACCUM,

    learning_rate=LEARNING_RATE,

    # T4 does not support BF16 reliably.
    # Therefore use FP16 on T4.
    fp16=True,

    bf16=False,

    logging_steps=5,

    save_strategy="epoch",

    save_total_limit=2,

    remove_unused_columns=False,

    report_to="none",

    dataloader_pin_memory=False,

    gradient_checkpointing=True,

    optim="paged_adamw_8bit",

    warmup_ratio=0.03,

    weight_decay=0.01,

    max_grad_norm=1.0,
)


trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    data_collator=collator,
)


print()
print("=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print()


trainer.train()


print()
print("=" * 60)
print("TRAINING COMPLETE ✅")
print("=" * 60)


STARTING TRAINING



/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Step,Training Loss
5,3.183200
10,1.641200
15,0.527300
20,0.325700
25,0.230500
30,0.238000
35,0.178100
40,0.179000
45,0.254100
50,0.225900


/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(



TRAINING COMPLETE ✅


In [ ]:
# ============================================================
# CELL 12 — SAVE LORA
# ============================================================

model.save_pretrained(
    LORA_DIR
)

processor.save_pretrained(
    LORA_DIR
)


print(
    "LoRA saved to:"
)

print(
    LORA_DIR
)

LoRA saved to:
/content/id-forgery-detector/outputs/lora_weights


In [ ]:
# ============================================================
# CELL 13 — PREDICTION
# ============================================================

def predict(model, processor, image_path):

    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                },
                {
                    "type": "text",
                    "text": QUESTION,
                },
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt",
        padding=True,
    )

    device = next(model.parameters()).device

    for key, value in inputs.items():
        if hasattr(value, "to"):
            inputs[key] = value.to(device)

    model.eval()

    with torch.inference_mode():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            use_cache=True,
        )

    input_length = inputs["input_ids"].shape[1]

    generated_ids = output_ids[0][input_length:]

    output = processor.decode(
        generated_ids,
        skip_special_tokens=True,
    )

    return output.strip()

In [ ]:
# ============================================================
# CELL 14 — CLASSIFICATION
# ============================================================

def classify_prediction(
    text,
):

    lower = text.lower()


    # Explicit negatives first
    if (
        "not forged" in lower
        or "not fake" in lower
        or "not a forgery" in lower
    ):

        return "authentic"


    if re.search(
        r"\bforged\b"
        r"|\bfake\b"
        r"|\btampered\b"
        r"|\btamper\b"
        r"|\bfraudulent\b",
        lower,
    ):

        return "forged"


    if re.search(
        r"\bauthentic\b"
        r"|\bgenuine\b"
        r"|\breal\b",
        lower,
    ):

        return "authentic"


    return "unknown"

In [ ]:
# ============================================================
# CELL 15 — EVALUATE MODEL
# ============================================================

test_samples = []

with open(DATASET_DIR / "test.jsonl") as f:

    for line in f:
        test_samples.append(
            json.loads(line)
        )


results = []

correct = 0


print(
    f"Testing on {len(test_samples)} images..."
)


for i, sample in enumerate(
    tqdm(test_samples, desc="Evaluating")
):

    try:

        prediction = predict(
            model,
            processor,
            PROJECT / sample["image"],
        )

        predicted_label = classify_prediction(
            prediction
        )

        true_label = sample["label"]

        is_correct = (
            predicted_label == true_label
        )

        if is_correct:
            correct += 1

        results.append({

            "image": sample["image"],

            "true_label": true_label,

            "predicted_label": predicted_label,

            "prediction": prediction,

            "correct": is_correct,

        })

    except Exception as e:

        print(
            f"\nERROR on image {i}: {sample['image']}"
        )

        print(
            type(e).__name__,
            str(e)
        )

        break


# ============================================================
# CALCULATE METRICS
# ============================================================

total = len(results)

accuracy = (
    correct / total
    if total > 0
    else 0
)


forged_results = [
    r for r in results
    if r["true_label"] == "forged"
]

authentic_results = [
    r for r in results
    if r["true_label"] == "authentic"
]


forged_correct = sum(
    r["correct"]
    for r in forged_results
)

authentic_correct = sum(
    r["correct"]
    for r in authentic_results
)


forged_recall = (
    forged_correct / len(forged_results)
    if forged_results
    else 0
)


authentic_recall = (
    authentic_correct / len(authentic_results)
    if authentic_results
    else 0
)


# ============================================================
# PRINT RESULTS
# ============================================================

print("\n")
print("=" * 60)
print("           ID FORGERY DETECTION RESULTS")
print("=" * 60)

print(
    f"Total evaluated:    {total}"
)

print(
    f"Correct predictions: {correct}"
)

print(
    f"Accuracy:           {accuracy:.2%}"
)

print(
    f"Forged recall:      {forged_recall:.2%}"
)

print(
    f"Authentic recall:   {authentic_recall:.2%}"
)

print("=" * 60)


# ============================================================
# SAVE RESULTS
# ============================================================

summary = {

    "accuracy": accuracy,

    "forged_recall": forged_recall,

    "authentic_recall": authentic_recall,

    "total_evaluated": total,

    "correct": correct,

    "results": results,

}


result_path = (
    OUTPUT_DIR /
    "eval_results.json"
)


result_path.write_text(
    json.dumps(
        summary,
        indent=2
    )
)


print(
    "\nResults saved to:",
    result_path
)

Testing on 30 images...


Evaluating:   0%|          | 0/30 [00:00<?, ?it/s]



           ID FORGERY DETECTION RESULTS
Total evaluated:    30
Correct predictions: 27
Accuracy:           90.00%
Forged recall:      93.33%
Authentic recall:   86.67%

Results saved to: /content/id-forgery-detector/outputs/eval_results.json


In [ ]:
# ============================================================
# CELL 17 — ZIP LORA WEIGHTS
# ============================================================

from google.colab import files


archive_path = shutil.make_archive(

    "/content/id_forgery_lora_weights",

    "zip",

    LORA_DIR,
)


print(
    "Created:",
    archive_path,
)


files.download(
    archive_path
)

Created: /content/id_forgery_lora_weights.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>